In [ ]:

# CONFIGURAÇÕES INICIAIS DAS ANÁLISES (PRESENTES EM TODOS OS SCRIPTS)
# IMPORTAR BIBLIOTECAS ---
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CAMINHOS ---
SCRIPT_DIR = Path(__file__).resolve().parent        # caminho desse script
ANALYTICS_DIR = SCRIPT_DIR.parent                   # pasta desse script

# PARA IMPORTAR FUNÇÕES DE EXTRAIR CSV ---
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))
from utils.export_utils import exportar_csv


# ROOT DO PROJETO ---
PROJECT_ROOT = Path(__file__).resolve().parents[3]

# ROOT DOS OUTPUTS ---
OUTPUT_DIR = (PROJECT_ROOT/ "scripts"/ "Analytics"/ "outputs"/ "gold_01")
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# SPARK ---
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 01 - Como está estruturado o mercado brasileiro de Dados? ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CAMINHO DO ARQUIVO ---
caminho_gold_01 = (PROJECT_ROOT/ "Gold"/ "perguntas_negocio"/ "gold_01_estrutura_mercado")

# BUSCA CSVs GERADOS PELO SPARK NA CRIAÇÃO DA GOLD ---
arquivos_gold_01 = [
    str(arquivo)
    for arquivo in caminho_gold_01.glob("part-*.csv")
]
print("Arquivos encontrados:")
print(arquivos_gold_01)

# CARREGAR GOLD 01 ---
df_estrutura = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_01)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INÍCIO DAS ANÁLISES ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# NÍVEL ---
# ANÁLISE INICIAL DA DIMENSÃO ---
df_nivel = (df_estrutura
    .filter(F.col("variavel") == "nivel")
    .orderBy("edicao", F.desc("pct_na_dimensao"))
)
df_nivel.show(100, truncate=False)
print('Qtd linhas', df_nivel.count())

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A dimensão de senioridade apresenta mudança de taxonomia na edição
2025-2026.

Nas edições 2023-2024 e 2024-2025 existem três categorias:
Júnior, Pleno e Sênior.

Em 2025-2026 foi adicionada a categoria Especialista/Staff+, que
representa 14,0% dos respondentes da edição.

Essa mudança impede a comparação direta das quatro categorias de
2025-2026 com as edições anteriores, pois parte da composição dos
níveis mais experientes passou a ser apresentada separadamente.

O número de respondentes também varia entre as edições:

- 2023-2024: 3.857 respondentes
- 2024-2025: 3.818 respondentes
- 2025-2026: 2.501 respondentes

Por esse motivo, a análise histórica será realizada por participação
percentual e somente entre períodos com taxonomia comparável.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# ESTRUTURA ATUAL POR NÍVEL ---
nivel_atual = (df_nivel
    .filter(F.col("edicao") == "2025-2026")
    .select("valor","contagem","pct_na_dimensao")
    .orderBy(F.desc("pct_na_dimensao"))
)
nivel_atual.show(truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Na edição 2025-2026, profissionais Sênior representam o maior grupo
da amostra, com 34,3% dos respondentes.

A distribuição por nível é:

- Sênior: 34,3%
- Pleno: 31,0%
- Júnior: 20,7%
- Especialista/Staff+: 14,0%

Sênior e Pleno são os níveis com maior representatividade na edição
mais recente.

A categoria Especialista/Staff+, introduzida nesta edição, representa
aproximadamente um em cada sete respondentes.

Os percentuais descrevem a composição da amostra da pesquisa e não
devem ser interpretados isoladamente como a distribuição de
senioridade de todo o mercado brasileiro de Dados.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# PARTICIPAÇÃO DE SÊNIOR + ESPECIALISTA/STAFF+ NA EDIÇÃO ATUAL ---
nivel_experiente_atual = (df_nivel
    .filter((F.col("edicao") == "2025-2026") & (F.col("valor").isin("Sênior", "Especialista/Staff+")))
    .agg(F.round(F.sum("pct_na_dimensao"),2).alias("pct_senior_staff"))
)
nivel_experiente_atual.show(truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Na edição 2025-2026, a soma das categorias Sênior e
Especialista/Staff+ corresponde a 48,3% dos respondentes.

Isso significa que quase metade da amostra atual está concentrada
nos dois níveis mais avançados de senioridade considerados nesta
classificação.

O indicador reforça a presença relevante de profissionais experientes
entre os respondentes da pesquisa.

PONTO DE ATENÇÃO:
Esse percentual é utilizado para descrever a edição atual.
Ele não deve ser comparado diretamente ao percentual de Sênior das
edições anteriores sem confirmar que Especialista/Staff+ representa
uma subdivisão equivalente da antiga categoria Sênior.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# COMPARAÇÃO HISTÓRICA ANTES DA MUDANÇA DE TAXONOMIA ---
comparativo_nivel_historico = (df_nivel
    .filter(F.col("edicao").isin("2023-2024","2024-2025"))
    .groupBy("valor")
    .pivot("edicao",["2023-2024", "2024-2025"])
    .agg(F.first("pct_na_dimensao"))
    .withColumn("variacao_pp",F.round(F.col("2024-2025") - F.col("2023-2024"),2))
    .orderBy(F.desc("variacao_pp"))
)
comparativo_nivel_historico.show(truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A comparação histórica foi realizada entre 2023-2024 e 2024-2025,
pois essas duas edições utilizam a mesma estrutura de categorias:
Júnior, Pleno e Sênior.

Entre os dois períodos, a participação de profissionais Sênior
aumentou de 36,8% para 41,2%, uma variação de +4,4 pontos percentuais.

A participação de profissionais Pleno permaneceu estável em 36,1%.

Já a participação de profissionais Júnior caiu de 27,1% para 22,7%,
uma redução de -4,4 pontos percentuais.

Portanto, entre as duas edições comparáveis, observa-se uma mudança
na composição da amostra em direção a maior participação de
profissionais Sênior, enquanto a participação de Pleno permanece
constante e a de Júnior diminui na mesma magnitude.

PONTO DE ATENÇÃO:
Essa mudança representa a composição dos respondentes das pesquisas
e não permite concluir, isoladamente, que houve redução de vagas
Júnior ou aumento de vagas Sênior no mercado brasileiro.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS PARA VISUALIZAÇÃO ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# NIVEL ATUAL.CSV ---
exportar_csv(nivel_atual,OUTPUT_DIR,"nivel_atual.csv")

# COMPARTIVO HISTORICO.CSV ---
exportar_csv(comparativo_nivel_historico,OUTPUT_DIR,"nivel_historico.csv")